# Build a simple agent

In this module, you will build and test a simple agent in the reimagined claims processing workflow - the **Authenticator Agent**. This agent will be used as a tool for the Orchestrator Agent in the Pre-Processing Phase

### Learning Objectives 
- Understand the Strands Agent class and how it wraps Bedrock Converse API
- Write a system prompt that encodes domain logic (insurance authentication rules)
- Pass structured claim data as input and get structured reasoning as output
- Observe the agent's reasoning trace (thinking steps)
- Understand when/why you'd graduate to AgentCore

#### Setup

We will use Amazon Nova 2 Lite reasoning mode. This lets the model allocate internal thinking tokens before producing a response. This is particularly effective for verfication tasks over multiple data sources, which is what the Authenticator agents does. 

In [ ]:
import boto3
import json
from strands import Agent
from strands.models import BedrockModel


#Nova 2 Lite ID
MODEL_ID = "us.amazon.nova-2-lite-v1:0"  
REGION   = "us-east-1"


# ── Nova 2 Lite with reasoning enabled ───────────────────────────────────────
# reasoningConfig is passed via additional_model_request_fields
# maxReasoningEffort: "low" | "medium" | "high"
# Use "medium" for authentication tasks — enough reasoning depth without
# the latency of "high" (reserve "high" for complex document analysis in NB04+)


#create a Strands BedrockModel provider instance:
model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature = 0.1,
    top_p=0.9,
    additional_request_fields={
        "reasoningConfig": {
            "type": "enabled",
            "maxReasoningEffort": "low" # note Temperature, topP and topK cannot be used if maxReasoningEffort is set to high
        }
    }
)
print(f"✅ Nova 2 Lite configured with reasoning enabled")
print(f"   Model: {MODEL_ID}")


#### Define mock policy data
In production, this comes from Amazin DynamoDB. Here, we use mock data to keep focus on the agent pattern

In [ ]:
MOCK_POLICY_RECORDS = {
    "POL-2024-001": {
        "policy_number": "POL-2024-001",
        "insured_name": "Robert Johnson",
        "insured_dob": "1955-03-15",
        "insured_ssn_last4": "[US_SOCIAL_SECURITY_NUMBER]",
        "beneficiary_name": "Mary Johnson",
        "beneficiary_relationship": "Spouse",
        "beneficiary_dob": "1958-07-22",
        "coverage_amount": 500000,
        "policy_status": "active",
        "policy_type": "whole_life"
    }

}

# Simulated claim submission (in production this will come from Amazon S3
SAMPLE_CLAIM_SUBMISSION = {
    "claim_id": "CLM-2025-00847",
    "policy_number": "POL-2024-001",
    "claimant_name": "Mary Johnson",
    "claimant_relationship": "Spouse",
    "claimant_dob": "1958-07-22",
    "date_of_death": "2025-01-10",
    "cause_of_death": "Natural causes",
    "documents_submitted": ["death_certificate", "claim_form", "beneficiary_id"],
    "required_documents": ["death_certificate", "claim_form", "beneficiary_id", "policy_document"]
}

print("✅ Mock data loaded")
print(f"Claim ID: {SAMPLE_CLAIM_SUBMISSION['claim_id']}")
print(f"Policy: {SAMPLE_CLAIM_SUBMISSION['policy_number']}")

#### Define the Authenticator Agent system prompt

In [ ]:
AUTHENTICATOR_SYSTEM_PROMPT = """You are the Authenticator Agent for end to end Insurance Claims processing worfklow.

Your role is to validate beneficiary identity and claim submission completeness 
before a claim proceeds to document processing.

## Your Validation Tasks

1. **Identity Verification**
   - Confirm claimant name matches the beneficiary on record
   - Verify relationship matches policy records
   - Check date of birth matches (if provided)
   - Flag any discrepancies for human review

"""

print("✅ System prompt defined")

#### Build the agent using strands and format input

In [ ]:
authenticator_agent = Agent(
    model=model,
    system_prompt=AUTHENTICATOR_SYSTEM_PROMPT,
)

def build_auth_request(claim: dict, policy_records: dict) -> str:
    """
    Formats claim submission + policy record into structured agent input.
    In production: policy record is a DynamoDB lookup by policy_number.
    """
    policy = policy_records.get(claim["policy_number"], {})
    missing_docs = set(claim["required_documents"]) - set(claim["documents_submitted"])

    return f"""## Authentication Request

**Claim ID:** {claim['claim_id']}
**Policy Number:** {claim['policy_number']}

### Claimant (from submission)
- Name: {claim['claimant_name']}
- Relationship: {claim['claimant_relationship']}
- Date of Birth: {claim['claimant_dob']}

### Beneficiary on Policy Record
- Name: {policy.get('beneficiary_name', 'NOT FOUND')}
- Relationship: {policy.get('beneficiary_relationship', 'NOT FOUND')}
- Date of Birth: {policy.get('beneficiary_dob', 'NOT FOUND')}
- Policy Status: {policy.get('policy_status', 'UNKNOWN')}

### Document Status
- Submitted: {', '.join(claim['documents_submitted'])}
- Required:  {', '.join(claim['required_documents'])}
- Missing:   {', '.join(missing_docs) if missing_docs else 'None'}

Please perform identity verification and completeness check.
"""

auth_request = build_auth_request(SAMPLE_CLAIM_SUBMISSION, MOCK_POLICY_RECORDS)
print(auth_request)
print("📋 Request prepared — ready to invoke agent")


#### Run the agent and inspect reasoning

In [ ]:
print("🤖 Running Authenticator Agent with Nova 2 Lite reasoning...")
print("=" * 60)

response = authenticator_agent(auth_request)

print("" + "=" * 30)
print("✅ Authentication complete")


#### Edge case: Test a mismatched claimant

Now we test what the agent does when identity of a claimant doesn't match. Review the reasoning trace as you test and see how rules in the system prompt are applied

In [ ]:
MISMATCHED_CLAIM = {
    "claim_id": "CLM-2025-00848",
    "policy_number": "POL-2024-001",
    "claimant_name": "James Wilson",              # ← Wrong name
    "claimant_relationship": "Business Partner",  # ← Wrong relationship
    "claimant_dob": "1970-05-01",
    "date_of_death": "2025-01-10",
    "cause_of_death": "Natural causes",
    "documents_submitted": ["death_certificate"],  # ← Incomplete
    "required_documents": ["death_certificate", "claim_form", "beneficiary_id", "policy_document"]
}

print("🔴 Testing mismatched claimant scenario...")
mismatch_request = build_auth_request(MISMATCHED_CLAIM, MOCK_POLICY_RECORDS)
mismatch_response = authenticator_agent(mismatch_request)


#### Review
- Compare the outputs from both runs.
-  What evidence did the model weigh differently?
- Why did it recommend 'Pass' vs 'Fail'?

## Notebook Complete

You've built a simple **Authenticator Agent** using the Strands `Agent` class with Nova 2 Lite's extended reasoning enabled.

### What You Built
- A Strands `Agent` with a domain-specific system prompt encoding insurance authentication rules
- Structured input formatting that combines claim submission data with policy records
- Nova 2 Lite reasoning traces that expose *how* the model evaluates identity evidence before reaching a verdict
- Edge case handling that demonstrates the agent's escalation logic for mismatched or incomplete claims

### What This Agent Cannot Do Yet

These gaps are intentional — each one motivates a future notebook:

| Limitation | Why It Matters | 
|---|---|
| **Mock policy lookup** | Real authentication requires querying a live policy system, not a hardcoded dict | 
| **No document inspection** | The agent trusts the claimant's word on submitted docs; it can't read them |
| **No PII protection** | DOB and names flow in plaintext — a HIPAA violation in production |
| **No session persistence** | Agent state is lost when the kernel restarts|
| **No structured handoff** | The agent returns text — nothing downstream acts on it automatically |
